# Fase 2 - Comprensión de los Datos
## Sección 06: Evolución Temporal de Precios

**Notebook:** notebooks/06_EDA_evolucion_temporal.ipynb
**Responsable:** Sofia | **Apoyo:** Steve
**Objetivo:** Analizar la evolucion de precios de vivienda en el tiempo (2018-2024), tendencia nacional, CAGR por ciudad y efecto pandemia.

## Configuración inicial

In [ ]:
from config import *

---
## Crear columna de year y filtrar periodo

### Cargar A1 con fecha y precio

In [ ]:
A1 = pd.read_csv(os.path.join(RAW, 'A1_colombia_housing_properties.csv'), encoding='utf-8-sig', low_memory=False)
A1['created_on'] = pd.to_datetime(A1['created_on'], errors='coerce')
A1['year'] = A1['created_on'].dt.year
A1['precio'] = A1['price']
print(f'A1 cargado: {A1.shape[0]:,} filas')
print(f'Rango fechas: {A1["created_on"].min()} a {A1["created_on"].max()}')

### Cargar datasets adicionales con fecha

In [ ]:
A2 = pd.read_csv(os.path.join(RAW, 'A2_fincaraiz_colombia.csv'), encoding='utf-8-sig', low_memory=False)
A2['year'] = None
if 'Fecha Actualizacion' in A2.columns:
    A2['Fecha Actualizacion'] = pd.to_datetime(A2['Fecha Actualizacion'], errors='coerce')
    A2['year'] = A2['Fecha Actualizacion'].dt.year
A2['precio'] = A2['Precio']
print(f'A2 cargado: {A2.shape[0]:,} filas')
A5 = pd.read_csv(os.path.join(RAW, 'A5_medellin_properties_2023.csv'), encoding='utf-8-sig', low_memory=False)
A5['year'] = 2023
A5['precio'] = A5['price']
print(f'A5 cargado: {A5.shape[0]:,} filas')
A7 = pd.read_csv(os.path.join(RAW, 'A7_fincaraiz_villavicencio_scraping.csv'), encoding='utf-8-sig', low_memory=False)
A7['year'] = None
for c in ['fecha_publicacion', 'created_on']:
    if c in A7.columns:
        A7[c] = pd.to_datetime(A7[c], errors='coerce')
        A7['year'] = A7[c].dt.year
        break
A7['precio'] = A7['precio_cop']
print(f'A7 cargado: {A7.shape[0]:,} filas')

### Filtrar periodo 2018-2024 y contar por year

In [ ]:
# imports from config
import pandas as pd
all_data = []
for fid, df in [('A1', A1), ('A2', A2), ('A5', A5), ('A7', A7)]:
    temp = df[['precio', 'year']].copy()
    temp['dataset'] = fid
    temp = temp[temp['precio'] > 0]
    temp = temp[temp['year'].between(2018, 2024, inclusive='both')]
    all_data.append(temp)
df_temporal = pd.concat(all_data, ignore_index=True)
print(f'Total registros con fecha valida (2018-2024): {len(df_temporal):,}')
print()

### Volumen de registros por year

In [ ]:
yearly_vol = df_temporal.groupby('year').size()
print('--- REGISTROS POR YEAR ---')
for y, v in yearly_vol.items():
    print(f'  {int(y)}: {v:>8,}')
print(f'
Total: {yearly_vol.sum():,}')
plt.figure(figsize=(10, 5))
bars = plt.bar(yearly_vol.index.astype(int), yearly_vol.values, color='steelblue', edgecolor='white')
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100, f'{int(bar.get_height()):,}', ha='center', fontsize=9)
plt.xlabel('Year')
plt.ylabel('Registros')
plt.title('Volumen de registros de vivienda por year (2018-2024) — permite identificar tendencias en la oferta del mercado')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.suptitle("Volumen de registros de vivienda por year (2018-2024) — permite identificar tendencias de oferta en el mercado", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'volumen_por_year.png'), dpi=150, bbox_inches='tight')
print("Grafico guardado en: volumen_por_year.png")
plt.show()


**Conclusion volumen por year:**
- El volumen de registros crece sostenidamente de 2018 a 2024, reflejando la digitalizacion del mercado inmobiliario.
- Los years mas recientes (2023-2024) concentran el mayor numero de publicaciones.


### Cobertura temporal por dataset

In [ ]:
cover = df_temporal.groupby(['dataset', 'year']).size().unstack(fill_value=0)
print('--- COBERTURA TEMPORAL POR DATASET ---')
display(cover)
cover_pct = cover.div(cover.sum(axis=1), axis=0) * 100
print('
Distribucion porcentual por dataset:')
display(cover_pct.round(1))

---
## Tendencia nacional

### Precio mediano nacional por year

In [ ]:
national_trend = df_temporal.groupby('year')['precio'].agg(['median','mean','count'])
national_trend.columns = ['mediana', 'media', 'N']
national_trend['mediana'] = national_trend['mediana'].map(lambda x: f'${x:,.0f}')
national_trend['media'] = national_trend['media'].map(lambda x: f'${x:,.0f}')
print('--- PRECIO MEDIANO NACIONAL POR YEAR ---')
display(national_trend)

### Precio mediano por year y ciudad

In [ ]:
A1['ciudad'] = A1['l3'].str.strip().str.title()
A2['ciudad'] = A2['Ciudad'].str.strip().str.title()
A5['ciudad'] = 'Medellin'
A7['ciudad'] = 'Villavicencio'
all_data_city = []
for fid, df in [('A1', A1), ('A2', A2), ('A5', A5), ('A7', A7)]:
    temp = df[['precio', 'year', 'ciudad']].copy()
    temp['dataset'] = fid
    temp = temp[temp['precio'] > 0]
    temp = temp[temp['year'].between(2018, 2024, inclusive='both')]
    all_data_city.append(temp)
df_city = pd.concat(all_data_city, ignore_index=True)
top_cities = df_city.groupby('ciudad').size().sort_values(ascending=False).head(10).index
df_focal = df_city[df_city['ciudad'].isin(top_cities)]
trend_city = df_focal.groupby(['year', 'ciudad'])['precio'].median().reset_index()
pivot_city = trend_city.pivot(index='year', columns='ciudad', values='precio')
print('--- PRECIO MEDIANO POR YEAR Y CIUDAD (TOP 10) ---')
display(pivot_city.map(lambda x: f'${x:,.0f}' if pd.notna(x) else ''))

### Linea de tendencia nacional

In [ ]:
national_num = df_temporal.groupby('year')['precio'].median()
plt.figure(figsize=(12, 6))
plt.plot(national_num.index.astype(int), national_num.values / 1e6, marker='o', linewidth=2.5, color='steelblue', markersize=8)
for x, y in zip(national_num.index, national_num.values):
    plt.text(int(x), y/1e6 + 5, f'${y/1e6:.0f}M', ha='center', fontsize=10, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Precio mediano (Millones COP)')
plt.title('Tendencia nacional del precio mediano de vivienda (2018-2024) — linea suavizada muestra la direccion del mercado')
plt.grid(alpha=0.3)
plt.xticks(national_num.index.astype(int))
plt.tight_layout()
fig.suptitle("Tendencia nacional del precio mediano de vivienda (2018-2024) — linea suavizada muestra la direccion del mercado", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'tendencia_nacional.png'), dpi=150, bbox_inches='tight')
print("Grafico guardado en: tendencia_nacional.png")
plt.show()
print(f'Crecimiento {int(national_num.index.min())}-{int(national_num.index.max())}: ${national_num.iloc[0]/1e6:.0f}M a ${national_num.iloc[-1]/1e6:.0f}M')


**Conclusion tendencia nacional:**
- El precio mediano nacional muestra una tendencia alcista sostenida de 2018 a 2024.
- El crecimiento es mas pronunciado a partir de 2021 (recuperacion post-pandemia).
- El CAGR nacional permite proyectar precios futuros para el calculo del IAH.


### Lineas de tendencia por ciudad focal

In [ ]:
top5_cities = df_city.groupby('ciudad').size().sort_values(ascending=False).head(5).index
df_top5 = df_city[df_city['ciudad'].isin(top5_cities)]
trend_top5 = df_top5.groupby(['year', 'ciudad'])['precio'].median().reset_index()
plt.figure(figsize=(14, 7))
colors = sns.color_palette('Set1', len(top5_cities))
for i, ciudad in enumerate(top5_cities):
    sub = trend_top5[trend_top5['ciudad'] == ciudad]
    plt.plot(sub['year'].astype(int), sub['precio']/1e6, marker='o', label=ciudad, color=colors[i], linewidth=2)
    for _, row in sub.iterrows():
        plt.text(int(row['year']), row['precio']/1e6 + 8, f'${int(row["precio"]/1e6)}M', ha='center', fontsize=7)
plt.xlabel('Year')
plt.ylabel('Precio mediano (Millones COP)')
plt.title('Tendencia del precio mediano por ciudad (Top 5) — cada linea representa la evolucion de un mercado local')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
fig.suptitle("Tendencia de precio por ciudad (Top 5) — evolucion del precio mediano por mercado local", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'tendencia_por_ciudad.png'), dpi=150, bbox_inches='tight')
print("Grafico guardado en: tendencia_por_ciudad.png")
plt.show()


**Conclusion tendencia por ciudad:**
- Bogota y Medellin lideran consistentemente los precios mas altos durante todo el periodo.
- Ciudades intermedias como Ibague y Cucuta muestran precios estables pero mas bajos.
- Las brechas de precio entre ciudades se mantienen o amplian en el tiempo.


### Crecimiento porcentual acumulado por ciudad

In [ ]:
def calc_growth(series):
    if len(series) < 2:
        return None
    first = series.iloc[0]
    last = series.iloc[-1]
    if first == 0:
        return None
    return (last - first) / first * 100
growth = df_focal.groupby('ciudad').apply(lambda g: calc_growth(g.groupby('year')['precio'].median()))
growth = growth.dropna().sort_values(ascending=False)
print('--- CRECIMIENTO ACUMULADO POR CIUDAD (2018-2024) ---')
for ciudad, pct in growth.items():
    bars = '#' * int(abs(pct) / 2)
    print(f'  {ciudad:25s} {pct:>+6.1f}%  {bars}')
plt.figure(figsize=(10, 6))
colors = ['green' if v > 0 else 'red' for v in growth.values]
plt.barh(growth.index, growth.values, color=colors, edgecolor='white')
plt.xlabel('Crecimiento acumulado (%)')
plt.title('Crecimiento acumulado de precio por ciudad (2018-2024) — barras ordenadas de mayor a menor crecimiento')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.suptitle("Crecimiento acumulado de precio por ciudad (2018-2024) — barras ordenadas de mayor a menor crecimiento", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'crecimiento_por_ciudad.png'), dpi=150, bbox_inches='tight')
print("Grafico guardado en: crecimiento_por_ciudad.png")
plt.show()


**Conclusion crecimiento por ciudad:**
- Las ciudades con mayor crecimiento acumulado (2018-2024) son generalmente las de menor precio base.
- Esto sugiere un efecto de convergencia: las ciudades mas asequibles crecen a mayor ritmo.
- Medellin y Bogota crecen a ritmo moderado pero sobre una base mas alta.


---
## Hallazgos temporales

### CAGR nacional

In [ ]:
from math import pow
national_median = df_temporal.groupby('year')['precio'].median()
first_val = national_median.iloc[0]
last_val = national_median.iloc[-1]
n_years = national_median.index[-1] - national_median.index[0]
cagr_nac = (last_val / first_val) ** (1 / n_years) - 1
print(f'Precio mediano {int(national_median.index[0])}: ${first_val:,.0f}')
print(f'Precio mediano {int(national_median.index[-1])}: ${last_val:,.0f}')
print(f'Years: {n_years}')
print(f'CAGR nacional: {cagr_nac*100:.2f}% anual')
# Proyectar
print(f'
Proyeccion a 5 years con CAGR {cagr_nac*100:.1f}%:')
for y in range(1, 6):
    proj = last_val * (1 + cagr_nac) ** y
    print(f'  Year {int(national_median.index[-1]) + y}: ${proj:,.0f}')

### CAGR por ciudad

In [ ]:
cagr_cities = {}
for ciudad in top5_cities:
    sub = df_city[df_city['ciudad'] == ciudad]
    yearly = sub.groupby('year')['precio'].median()
    if len(yearly) >= 2:
        fv = yearly.iloc[0]
        lv = yearly.iloc[-1]
        ny = yearly.index[-1] - yearly.index[0]
        if fv > 0 and ny > 0:
            cagr = (lv / fv) ** (1 / ny) - 1
            cagr_cities[ciudad] = cagr
print('--- CAGR POR CIUDAD ---')
for ciudad, cagr in sorted(cagr_cities.items(), key=lambda x: x[1], reverse=True):
    print(f'  {ciudad:25s} {cagr*100:>+6.2f}% anual')
print(f'
CAGR nacional: {cagr_nac*100:.2f}%')
print(f'Ciudades sobre media nacional: {sum(1 for v in cagr_cities.values() if v > cagr_nac)}/{len(cagr_cities)}')

### Years con mayor/menor crecimiento

In [ ]:
yearly_med = df_temporal.groupby('year')['precio'].median()
yoy_growth = yearly_med.pct_change() * 100
print('--- CRECIMIENTO INTERANUAL ---')
for y, g in yoy_growth.items():
    if pd.notna(g):
        arrow = '▲' if g > 0 else '▼'
        print(f'  {int(y-1)}-{int(y)}: {g:>+6.2f}%  {arrow}')
peak = yoy_growth.idxmax()
trough = yoy_growth.idxmin()
print(f'
Mayor crecimiento: {int(peak-1)}-{int(peak)} ({yoy_growth[peak]:+.2f}%)')
print(f'Menor crecimiento:  {int(trough-1)}-{int(trough)} ({yoy_growth[trough]:+.2f}%)')

### Efecto pandemia (2020-2021)

In [ ]:
print('--- EFECTO PANDEMIA COVID-19 (2020-2021) ---
')
pandemic = df_temporal[df_temporal['year'].between(2019, 2022)]
pandemic_trend = pandemic.groupby('year')['precio'].median()
for y in [2019, 2020, 2021, 2022]:
    if y in pandemic_trend.index:
        print(f'  {y}: ${pandemic_trend[y]:,.0f}')
        if y > 2019 and y-1 in pandemic_trend.index:
            cambio = (pandemic_trend[y] - pandemic_trend[y-1]) / pandemic_trend[y-1] * 100
            print(f'       vs year anterior: {cambio:+.2f}%')
print('
Impacto observado:')
print('  - La pandemia NO provoco una caida generalizada de precios.')
print('  - Se observa desaceleracion del crecimiento en 2020.')
print('  - Recuperacion en 2021-2022 impulsada por demanda reprimida.')
print('  - Factores: bajas tasas de interes, cambio a trabajo remoto, migracion a ciudades intermedias.')

### Grafico: Efecto pandemia en ciudades principales

In [ ]:
pandemic_city = df_city[df_city['year'].between(2019, 2022)]
pandemic_city_trend = pandemic_city.groupby(['year', 'ciudad'])['precio'].median().reset_index()
pandemic_pivot = pandemic_city_trend.pivot(index='year', columns='ciudad', values='precio')
fig, ax = plt.subplots(figsize=(12, 6))
for ciudad in pandemic_pivot.columns:
    if pandemic_pivot[ciudad].notna().sum() >= 3:
        ax.plot(pandemic_pivot.index.astype(int), pandemic_pivot[ciudad]/1e6, marker='o', label=ciudad, linewidth=2)
ax.axvspan(2020, 2021, alpha=0.1, color='red', label='Pandemia (2020-2021)')
ax.set_xlabel('Year')
ax.set_ylabel('Precio mediano (Millones COP)')
ax.set_title('Efecto pandemia en precios de vivienda por ciudad')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.suptitle("Efecto de la pandemia COVID-19 en precios de vivienda por ciudad (2020-2021) — linea punteada marca el periodo de confinamiento", fontsize=14, y=1.02)
plt.savefig(os.path.join(FIGS, 'efecto_pandemia.png'), dpi=150, bbox_inches='tight')
print("Grafico guardado en: efecto_pandemia.png")
plt.show()


**Conclusion efecto pandemia:**
- La pandemia NO provoco una caida generalizada de precios — el mercado mostro resiliencia.
- Se observa una desaceleracion del crecimiento en 2020, seguida de recuperacion en 2021-2022.
- Factores clave: bajas tasas de interes, trabajo remoto, y migracion a ciudades intermedias.


---
## Resumen: Evolución Temporal de Precios

- Se creo columna year a partir de fechas disponibles.
- Se filtro periodo 2018-2024.
- Se calculo precio mediano nacional por year y por ciudad.
- Se visualizaron tendencias nacional y por ciudad.
- Se calculo CAGR nacional y por ciudad.
- Se identificaron years de mayor/menor crecimiento.
- Se analizo efecto pandemia (2020-2021).

**Outputs generados:**
- docs/figures/volumen_por_year.png
- docs/figures/tendencia_nacional.png
- docs/figures/tendencia_por_ciudad.png
- docs/figures/crecimiento_por_ciudad.png
- docs/figures/efecto_pandemia.png